# NPC Engine — Phase 0

**Goal:** prove a diffusion LM (Dream 7B Instruct) can do in-character NPC dialogue with parallel candidate generation + confidence ranking, at interactive latency, on one Colab GPU.

What to watch for:
- **Latency**: p50 per turn (8 candidates denoised in one batched pass)
- **Confidence**: does the top candidate's score feel honest? Do weak lines score low?
- **Fallbacks**: low-confidence turns ship a canned line instead of a broken one
- **Diversity**: are the 8 candidates actually different lines?


In [ ]:
!pip install -q "transformers==4.53.3" accelerate bitsandbytes

## Restore the `npc_engine` package (embedded in this notebook)

In [ ]:
import base64, pathlib
files = {
    "npc_engine/__init__.py": "IiIiTlBDIEVuZ2luZSBQaGFzZSAwIOKAlCBhIHRpbnkgdGV4dC1kaWZmdXNpb24gTlBDIGRpYWxvZ3VlIGhhcm5lc3MuCgpCdWlsdCBvbiBEcmVhbSA3QiBJbnN0cnVjdCAoRHJlYW0tb3JnL0RyZWFtLXYwLUluc3RydWN0LTdCKSwgYSBtYXNrZWQgZGlmZnVzaW9uIExNLgpUaGUgZGlmZnVzaW9uLW5hdGl2ZSBmZWF0dXJlcyB1c2VkIGhlcmU6CgotIHBhcmFsbGVsIGNhbmRpZGF0ZSBnZW5lcmF0aW9uOiBOIG1hc2tlZCBjYW52YXNlcyBkZW5vaXNlZCBpbiBPTkUgYmF0Y2hlZCBwYXNzCiAgKEFSIHdvdWxkIG5lZWQgTiBzZXF1ZW50aWFsIGRlY29kZXMpCi0gY29uZmlkZW5jZSByYW5raW5nOiBwZXItdG9rZW4gbWF4LXByb2Igb3ZlciB0aGUgZ2VuZXJhdGVkIHNwYW4sIGZyb20gYSBzaW5nbGUKICBzY29yaW5nIGZvcndhcmQgcGFzcyDigJQgbm8gc2Vjb25kIG1vZGVsCi0gY29uZmlkZW5jZSBmYWxsYmFjazogaWYgdGhlIGJlc3QgY2FuZGlkYXRlIHNjb3JlcyBiZWxvdyB0aHJlc2hvbGQsIHRoZSBlbmdpbmUKICBzaGlwcyBhIHNhZmUgY2FubmVkIGxpbmUgaW5zdGVhZCBvZiBhIGJyb2tlbiBvbmUgKHRoZSBhbnRpLWdhbWVzbG9wIGd1YXJkKQoiIiIKCmZyb20gLnBlcnNvbmFzIGltcG9ydCBQZXJzb25hLCBTVEFSVEVSX1BFUlNPTkFTLCBzeXN0ZW1fcHJvbXB0X2Zvcgpmcm9tIC5lbmdpbmUgaW1wb3J0IE5QQ0VuZ2luZSwgTlBDUmVzcG9uc2UKCl9fdmVyc2lvbl9fID0gIjAuMi4xIgoKX19hbGxfXyA9IFsKICAgICJQZXJzb25hIiwgIlNUQVJURVJfUEVSU09OQVMiLCAic3lzdGVtX3Byb21wdF9mb3IiLAogICAgIk5QQ0VuZ2luZSIsICJOUENSZXNwb25zZSIsCl0K",
    "npc_engine/personas.py": "IiIiUGVyc29uYSBjYXJkcyBmb3IgdGhlIE5QQyBFbmdpbmUuCgpBIHBlcnNvbmEgY2FyZCBpcyB0aGUgY29uZGl0aW9uaW5nIGNvbnRleHQgdGhlIGRpZmZ1c2lvbiBtb2RlbCBzZWVzIHdoZW4gaXQKZGVub2lzZXMgYW4gTlBDIGxpbmU6IHdobyB0aGUgY2hhcmFjdGVyIGlzLCBob3cgdGhleSB0YWxrLCB3aGF0IHRoZXkga25vdy4KVGhlIGNhcmQgc3RheXMgQ0xFQU4gKG5ldmVyIG1hc2tlZCkg4oCUIG9ubHkgdGhlIE5QQydzIGxpbmUgaXMgZGVub2lzZWQuCiIiIgoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZAoKClJVTEVTID0gKAogICAgIlJ1bGVzOiBzdGF5IGluIGNoYXJhY3RlciBhdCBhbGwgdGltZXMuIFJlcGx5IHdpdGggMS0yIHNob3J0IHNwb2tlbiAiCiAgICAibGluZXMgb25seSAodW5kZXIgNDAgd29yZHMpLiBOZXZlciBicmVhayBjaGFyYWN0ZXIsIG5ldmVyIG1lbnRpb24gIgogICAgImJlaW5nIGFuIEFJLiIKKQoKCmRlZiBzeXN0ZW1fcHJvbXB0X2ZvcihuYW1lOiBzdHIsIGNhcmQ6IHN0ciwgd29ybGRfc3RhdGU6IHN0ciwKICAgICAgICAgICAgICAgICAgICAgIGhpc3Rvcnk6IGxpc3QgfCBOb25lID0gTm9uZSkgLT4gc3RyOgogICAgIiIiVEhFIGNhbm9uaWNhbCBOUEMgc3lzdGVtIHByb21wdCDigJQgdXNlZCBieSB0aGUgZW5naW5lIGF0IGluZmVyZW5jZSBBTkQKICAgIGJ5IHRoZSBmaW5lLXR1bmUgZGF0YSBidWlsZGVyLCBzbyB0cmFpbiBhbmQgaW5mZXJlbmNlIGZvcm1hdHMgbWF0Y2guIiIiCiAgICBwYXJ0cyA9IFtmIllvdSBhcmUge25hbWV9LiB7Y2FyZH0iXQogICAgaWYgaGlzdG9yeToKICAgICAgICBwYXJ0cy5hcHBlbmQoIlJlY2VudCBjb252ZXJzYXRpb246XG4iICsgIlxuIi5qb2luKGhpc3RvcnkpKQogICAgcGFydHMuYXBwZW5kKGYiQ3VycmVudCBzaXR1YXRpb246IHt3b3JsZF9zdGF0ZX0iKQogICAgcGFydHMuYXBwZW5kKFJVTEVTKQogICAgcmV0dXJuICJcblxuIi5qb2luKHBhcnRzKQoKCkBkYXRhY2xhc3MKY2xhc3MgUGVyc29uYToKICAgIG5hbWU6IHN0cgogICAgcm9sZTogc3RyCiAgICBwZXJzb25hbGl0eTogc3RyCiAgICBzcGVlY2hfc3R5bGU6IHN0cgogICAga25vd2xlZGdlOiBzdHIKICAgICMgU2FmZSBjYW5uZWQgbGluZXMgdXNlZCB3aGVuIG1vZGVsIGNvbmZpZGVuY2UgaXMgYmVsb3cgdGhyZXNob2xkLgogICAgZmFsbGJhY2tfbGluZXM6IGxpc3QgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKCiAgICBkZWYgY2FyZF90ZXh0KHNlbGYpIC0+IHN0cjoKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICBmIntzZWxmLnJvbGV9LiBQZXJzb25hbGl0eToge3NlbGYucGVyc29uYWxpdHl9LiAiCiAgICAgICAgICAgIGYiU3BlZWNoIHN0eWxlOiB7c2VsZi5zcGVlY2hfc3R5bGV9LiAiCiAgICAgICAgICAgIGYiV2hhdCB5b3Uga25vdzoge3NlbGYua25vd2xlZGdlfS4iCiAgICAgICAgKQoKICAgIGRlZiBzeXN0ZW1fcHJvbXB0KHNlbGYsIHdvcmxkX3N0YXRlOiBzdHIsIGhpc3Rvcnk6IGxpc3QgfCBOb25lID0gTm9uZSkgLT4gc3RyOgogICAgICAgIHJldHVybiBzeXN0ZW1fcHJvbXB0X2ZvcihzZWxmLm5hbWUsIHNlbGYuY2FyZF90ZXh0KCksIHdvcmxkX3N0YXRlLCBoaXN0b3J5KQoKClNUQVJURVJfUEVSU09OQVMgPSBbCiAgICBQZXJzb25hKAogICAgICAgIG5hbWU9IkJvcmluIElyb25maXN0IiwKICAgICAgICByb2xlPSJ0aGUgd2VhcnkgYmxhY2tzbWl0aCBvZiBNaWxsYnJvb2sgdmlsbGFnZSIsCiAgICAgICAgcGVyc29uYWxpdHk9ImdydWZmIGJ1dCBraW5kLCB0aXJlZCwgZHJ5IGh1bW9yLCBwcm90ZWN0aXZlIG9mIHRoZSB2aWxsYWdlIiwKICAgICAgICBzcGVlY2hfc3R5bGU9InNob3J0IGNsaXBwZWQgc2VudGVuY2VzLCBjYWxscyBwZW9wbGUgJ2xhZCcgb3IgJ2xhc3MnLCBvY2Nhc2lvbmFsIGZvcmdlIG1ldGFwaG9ycyIsCiAgICAgICAga25vd2xlZGdlPSJ3ZWFwb25zLCBhcm1vciwgdGhlIG1pbGxlcidzIGJyb2tlbiBjYXJ0LCBydW1vcnMgb2YgYmFuZGl0cyBvbiB0aGUgbm9ydGggcm9hZCIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJBeWUuIiwgIkhtbS4gQ2FuJ3Qgc2F5LiIsICJTdGVlbCdzIGhvbmVzdC4gUGVvcGxlIGxlc3Mgc28uIl0sCiAgICApLAogICAgUGVyc29uYSgKICAgICAgICBuYW1lPSJNaXJhIFRoaXN0bGVkb3duIiwKICAgICAgICByb2xlPSJ0aGUgY2hlZXJmdWwgaW5ua2VlcGVyIG9mIHRoZSBSdXN0eSBLZXR0bGUiLAogICAgICAgIHBlcnNvbmFsaXR5PSJ3YXJtLCBjaGF0dHksIGdvc3NpcHksIG1vdGhlcmx5IHRvIHRyYXZlbGVycyIsCiAgICAgICAgc3BlZWNoX3N0eWxlPSJsb25nZXIgZmxvd2luZyBzZW50ZW5jZXMsIGV4Y2xhbWF0aW9ucywgYXNrcyBxdWVzdGlvbnMgYmFjaywgZm9vZCBtZXRhcGhvcnMiLAogICAgICAgIGtub3dsZWRnZT0idmlsbGFnZSBnb3NzaXAsIHJvb20gcmF0ZXMgKDUgY29wcGVyIGEgbmlnaHQpLCB0aGUgc3RldyByZWNpcGUsIHdobyBvd2VzIGhlciBtb25leSIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJPaCwgZGVhcmllISIsICJOb3cgdGhhdCdzIGEgdGFsZS4iLCAiU2l0LCBzaXQhIFN0ZXcncyBvbi4iXSwKICAgICksCiAgICBQZXJzb25hKAogICAgICAgIG5hbWU9IkNhcHRhaW4gU2VyYSBWb3NzIiwKICAgICAgICByb2xlPSJjYXB0YWluIG9mIHRoZSB2aWxsYWdlIGd1YXJkIiwKICAgICAgICBwZXJzb25hbGl0eT0iZGlzY2lwbGluZWQsIHN1c3BpY2lvdXMgb2Ygc3RyYW5nZXJzLCBzZWNyZXRseSB3b3JyaWVkIGFib3V0IHRoZSBiYW5kaXRzIiwKICAgICAgICBzcGVlY2hfc3R5bGU9InRlcnNlLCBkaXJlY3QsIG1pbGl0YXJ5IHBocmFzaW5nLCBubyB3YXN0ZWQgd29yZHMiLAogICAgICAgIGtub3dsZWRnZT0iZ3VhcmQgcm90YXRpb25zLCB0aGUgbm9ydGggcm9hZCBhbWJ1c2ggbGFzdCB3ZWVrLCB0aGUgbWF5b3IncyBvcmRlcnMiLAogICAgICAgIGZhbGxiYWNrX2xpbmVzPVsiU3RhdGUgeW91ciBidXNpbmVzcy4iLCAiTm90ZWQuIiwgIk1vdmUgYWxvbmcuIl0sCiAgICApLAogICAgUGVyc29uYSgKICAgICAgICBuYW1lPSJGZWxpeCBRdWlja2ZpbmdlcnMiLAogICAgICAgIHJvbGU9ImEgc2hhZHkgdHJhdmVsaW5nIG1lcmNoYW50IiwKICAgICAgICBwZXJzb25hbGl0eT0ib2lseSBjaGFybSwgYWx3YXlzIHNlbGxpbmcsIG5lcnZvdXMgd2hlbiBhc2tlZCBhYm91dCBoaXMgc3VwcGxpZXJzIiwKICAgICAgICBzcGVlY2hfc3R5bGU9ImZhc3QgcGF0dGVyLCBzdXBlcmxhdGl2ZXMsICdteSBmcmllbmQnLCBkZWZsZWN0cyBxdWVzdGlvbnMgd2l0aCBvZmZlcnMiLAogICAgICAgIGtub3dsZWRnZT0iaGlzIHdhcmVzIChwb3Rpb25zLCBtYXBzLCAnZ2VudWluZScgcmVsaWNzKSwgdGhlIHJvYWRzIGJldHdlZW4gdG93bnMsIHByaWNlcyIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJNeSBmcmllbmQhIiwgIkZvciB5b3UsIGEgc3BlY2lhbCBwcmljZS4iLCAiQWgsIGJ1c2luZXNzIGlzIGJ1c2luZXNzLiJdLAogICAgKSwKICAgIFBlcnNvbmEoCiAgICAgICAgbmFtZT0iRWxkZXIgV3JlbiIsCiAgICAgICAgcm9sZT0idGhlIHZpbGxhZ2UgaGVhbGVyIGFuZCBoZXJiYWxpc3QiLAogICAgICAgIHBlcnNvbmFsaXR5PSJjYWxtLCB3aXNlLCBzcGVha3MgaW4gcGxhbnQgbWV0YXBob3JzLCBnZW50bGUgYnV0IGZpcm0iLAogICAgICAgIHNwZWVjaF9zdHlsZT0ic2xvdyBtZWFzdXJlZCBzcGVlY2gsIHJoZXRvcmljYWwgcXVlc3Rpb25zLCByZWZlcmVuY2VzIHRvIGhlcmJzIGFuZCBzZWFzb25zIiwKICAgICAgICBrbm93bGVkZ2U9ImhlYWxpbmcgaGVyYnMsIHRoZSBmZXZlciBnb2luZyBhcm91bmQsIHBvdWx0aWNlcywgb2xkIHZpbGxhZ2UgaGlzdG9yeSIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJUaGUgYm9keSBrbm93cy4iLCAiUmVzdCwgY2hpbGQuIiwgIk5hdHVyZSBwcm92aWRlcy4iXSwKICAgICksCiAgICBQZXJzb25hKAogICAgICAgIG5hbWU9IlBpcCBVbmRlcmJvdWdoIiwKICAgICAgICByb2xlPSJhIG5lcnZvdXMgeW91bmcgbWVzc2VuZ2VyIiwKICAgICAgICBwZXJzb25hbGl0eT0iYW54aW91cywgZWFnZXIgdG8gcGxlYXNlLCBvdXQgb2YgYnJlYXRoLCB0YWxrcyB0b28gZmFzdCB3aGVuIHNjYXJlZCIsCiAgICAgICAgc3BlZWNoX3N0eWxlPSJmcmFnbWVudHMsIHN0YW1tZXJpbmcgd2hlbiBuZXJ2b3VzLCByZXBlYXRzIGhpbXNlbGYiLAogICAgICAgIGtub3dsZWRnZT0idGhlIHJvYWRzLCB3aG8gc2VudCBoaW0sIHRoZSBzZWFsZWQgbGV0dGVyIGhlIGNhcnJpZXMgKGhlIGhhc24ndCByZWFkIGl0KSIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJJ4oCUIEkganVzdCBkZWxpdmVyISIsICJQbGVhc2UgZG9uJ3Qgc2hvb3QgdGhlIG1lc3NlbmdlciEiLCAiR290dGEgcnVuISJdLAogICAgKSwKXQo=",
    "npc_engine/engine.py": "IiIiTlBDRW5naW5lIOKAlCBwYXJhbGxlbCBjYW5kaWRhdGUgZ2VuZXJhdGlvbiArIGNvbmZpZGVuY2UgcmFua2luZyBmb3IgTlBDIGxpbmVzLgoKSG93IGl0IHdvcmtzIChkaWZmdXNpb24td2lzZSk6CjEuIFRoZSBwZXJzb25hIGNhcmQgKyB3b3JsZCBzdGF0ZSArIHBsYXllciBsaW5lIGZvcm0gdGhlIHByb21wdC4gVGhlIHByb21wdCBpcwogICBrZXB0IGZ1bGx5IGNsZWFuIChuZXZlciBtYXNrZWQpLgoyLiBXZSBhcHBlbmQgYSBjYW52YXMgb2YgW01BU0tdIHRva2VucyBhbmQgZXhwYW5kIGl0IHRvIGEgYmF0Y2ggb2YgTiBjYW52YXNlcy4KICAgQWxsIE4gY2FuZGlkYXRlcyBhcmUgZGVub2lzZWQgaW4gT05FIGJhdGNoZWQgZGlmZnVzaW9uIHBhc3Mg4oCUIEFSIHdvdWxkIG5lZWQKICAgTiBzZXF1ZW50aWFsIGdlbmVyYXRpb25zLiBEaXZlcnNpdHkgY29tZXMgZnJvbSB0ZW1wZXJhdHVyZS90b3AtcCBzYW1wbGluZy4KMy4gQSBzaW5nbGUgc2NvcmluZyBmb3J3YXJkIHBhc3MgY29tcHV0ZXMsIGZvciBlYWNoIGNhbmRpZGF0ZSwgdGhlIG1lYW4KICAgbWF4LXByb2JhYmlsaXR5IG92ZXIgaXRzIGdlbmVyYXRlZCB0b2tlbnMgKGNvbmZpZGVuY2UpIHBsdXMgdGhlIHdlYWtlc3QKICAgdG9rZW4ncyBwcm9iYWJpbGl0eS4gQ2FuZGlkYXRlcyBhcmUgcmFua2VkIGJ5IGNvbmZpZGVuY2UuCjQuIElmIHRoZSBiZXN0IGNhbmRpZGF0ZSdzIGNvbmZpZGVuY2UgaXMgYmVsb3cgdGhlIHRocmVzaG9sZCwgdGhlIGVuZ2luZQogICByZXR1cm5zIG9uZSBvZiB0aGUgcGVyc29uYSdzIGNhbm5lZCBmYWxsYmFjayBsaW5lcyBpbnN0ZWFkIChmbGFnZ2VkKSwgc28gYQogICBsb3ctY29uZmlkZW5jZSBnZW5lcmF0aW9uIG5ldmVyIHJlYWNoZXMgdGhlIHBsYXllci4KIiIiCgppbXBvcnQgdGltZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKZnJvbSAucGVyc29uYXMgaW1wb3J0IFBlcnNvbmEKCkRFRkFVTFRfTU9ERUwgPSAiRHJlYW0tb3JnL0RyZWFtLXYwLUluc3RydWN0LTdCIgpNQVNLX1RPS0VOID0gIjx8bWFza3w+IgoKCkBkYXRhY2xhc3MKY2xhc3MgTlBDUmVzcG9uc2U6CiAgICB0ZXh0OiBzdHIgICAgICAgICAgICAgICAgICMgdGhlIGxpbmUgdGhlIGdhbWUgc2hvdWxkIHNoaXAKICAgIGNvbmZpZGVuY2U6IGZsb2F0ICAgICAgICAgIyBjb25maWRlbmNlIG9mIHRoZSBzaGlwcGVkIGxpbmUgKDEuMCBmb3IgZmFsbGJhY2spCiAgICB1c2VkX2ZhbGxiYWNrOiBib29sICAgICAgICMgVHJ1ZSBpZiB0aGUgbW9kZWwgd2FzIHRvbyB1bnN1cmUgYW5kIHdlIGNhbm5lZCBpdAogICAgY2FuZGlkYXRlczogbGlzdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KSAgIyBbKHRleHQsIGNvbmZpZGVuY2UpLCAuLi5dIHJhbmtlZAogICAgbGF0ZW5jeV9tczogZmxvYXQgPSAwLjAgICAjIHdhbGwgdGltZSBmb3IgdGhlIHdob2xlIHJlc3BvbmQoKSBjYWxsCgoKY2xhc3MgTlBDRW5naW5lOgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgbW9kZWxfaWQ6IHN0ciA9IERFRkFVTFRfTU9ERUwsCiAgICAgICAgZGV2aWNlOiBzdHIgfCBOb25lID0gTm9uZSwKICAgICAgICBsb2FkX2luXzhiaXQ6IGJvb2wgPSBGYWxzZSwKICAgICAgICB0b3JjaF9kdHlwZTogdG9yY2guZHR5cGUgfCBOb25lID0gTm9uZSwKICAgICAgICBhZGFwdGVyX3BhdGg6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgKToKICAgICAgICAiIiJMb2FkIHRoZSBkaWZmdXNpb24gTE0uIE9uIGEgMTZHQiBHUFUgdXNlIGxvYWRfaW5fOGJpdD1UcnVlOyBvbiAyNEdCKwogICAgICAgIGZwMTYvYmYxNiBpcyBmaW5lLiBhZGFwdGVyX3BhdGggbG9hZHMgYSBMb1JBIGFkYXB0ZXIgb24gdG9wLiIiIgogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWwsIEF1dG9Ub2tlbml6ZXIKCiAgICAgICAgc2VsZi5kZXZpY2UgPSBkZXZpY2Ugb3IgKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICAgICAga3dhcmdzID0geyJ0cnVzdF9yZW1vdGVfY29kZSI6IFRydWV9CiAgICAgICAgaWYgbG9hZF9pbl84Yml0OgogICAgICAgICAgICAjIE1vZGVybiB0cmFuc2Zvcm1lcnM6IDgtYml0IGxvYWRpbmcgZ29lcyB0aHJvdWdoIGEgcXVhbnRpemF0aW9uCiAgICAgICAgICAgICMgY29uZmlnICsgZGV2aWNlX21hcC4gUGFzc2luZyBsb2FkX2luXzhiaXQ9VHJ1ZSBkaXJlY3RseSBsZWFrcyB0aGUKICAgICAgICAgICAgIyBrd2FyZyBpbnRvIHRoZSBjdXN0b20gbW9kZWwncyBfX2luaXRfXyBhbmQgY3Jhc2hlcyBpdC4KICAgICAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEJpdHNBbmRCeXRlc0NvbmZpZwogICAgICAgICAgICBrd2FyZ3NbInF1YW50aXphdGlvbl9jb25maWciXSA9IEJpdHNBbmRCeXRlc0NvbmZpZyhsb2FkX2luXzhiaXQ9VHJ1ZSkKICAgICAgICAgICAga3dhcmdzWyJkZXZpY2VfbWFwIl0gPSAiYXV0byIKICAgICAgICBlbHNlOgogICAgICAgICAgICBrd2FyZ3NbInRvcmNoX2R0eXBlIl0gPSB0b3JjaF9kdHlwZSBvciB0b3JjaC5mbG9hdDE2CiAgICAgICAgc2VsZi5tb2RlbCA9IEF1dG9Nb2RlbC5mcm9tX3ByZXRyYWluZWQobW9kZWxfaWQsICoqa3dhcmdzKQogICAgICAgIGlmIG5vdCBsb2FkX2luXzhiaXQ6CiAgICAgICAgICAgIHNlbGYubW9kZWwgPSBzZWxmLm1vZGVsLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIGlmIGFkYXB0ZXJfcGF0aDoKICAgICAgICAgICAgZnJvbSBwZWZ0IGltcG9ydCBQZWZ0TW9kZWwKICAgICAgICAgICAgc2VsZi5tb2RlbCA9IFBlZnRNb2RlbC5mcm9tX3ByZXRyYWluZWQoc2VsZi5tb2RlbCwgYWRhcHRlcl9wYXRoKQogICAgICAgIHNlbGYubW9kZWwuZXZhbCgpCiAgICAgICAgc2VsZi50b2tlbml6ZXIgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChtb2RlbF9pZCwgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSkKICAgICAgICBzZWxmLm1hc2tfdG9rZW5faWQgPSBzZWxmLnRva2VuaXplci5jb252ZXJ0X3Rva2Vuc190b19pZHMoTUFTS19UT0tFTikKICAgICAgICBlb3MgPSBzZWxmLnRva2VuaXplci5lb3NfdG9rZW5faWQKICAgICAgICBzZWxmLmVvc19pZHMgPSBzZXQoZW9zIGlmIGlzaW5zdGFuY2UoZW9zLCBsaXN0KSBlbHNlIFtlb3NdKQoKICAgICMgLS0tLSBwcm9tcHQgYnVpbGRpbmcgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIF9wcm9tcHRfaWRzKHNlbGYsIHBlcnNvbmE6IFBlcnNvbmEsIHdvcmxkX3N0YXRlOiBzdHIsIHBsYXllcl9saW5lOiBzdHIsCiAgICAgICAgICAgICAgICAgICAgaGlzdG9yeTogbGlzdCB8IE5vbmUgPSBOb25lKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgbWVzc2FnZXMgPSBbCiAgICAgICAgICAgIHsicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6IHBlcnNvbmEuc3lzdGVtX3Byb21wdCh3b3JsZF9zdGF0ZSwgaGlzdG9yeSl9LAogICAgICAgICAgICB7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogcGxheWVyX2xpbmV9LAogICAgICAgIF0KICAgICAgICBpZHMgPSBzZWxmLnRva2VuaXplci5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgICAgICBtZXNzYWdlcywgdG9rZW5pemU9VHJ1ZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUKICAgICAgICApCiAgICAgICAgcmV0dXJuIHRvcmNoLnRlbnNvcihbaWRzXSwgZHR5cGU9dG9yY2gubG9uZywgZGV2aWNlPXNlbGYuZGV2aWNlKQoKICAgICMgLS0tLSBjb25maWRlbmNlIHNjb3JpbmcgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIF9zY29yZV9jYW5kaWRhdGVzKHNlbGYsIHNlcXM6IHRvcmNoLlRlbnNvciwgcHJvbXB0X2xlbjogaW50KToKICAgICAgICAiIiJPbmUgZm9yd2FyZCBwYXNzOyBwZXIgY2FuZGlkYXRlLCAobWVhbiBtYXgtcHJvYiwgbWluIG1heC1wcm9iKSBvdmVyCiAgICAgICAgdGhlIGdlbmVyYXRlZCBzcGFuICh1cCB0byBpdHMgZmlyc3QgRU9TKS4iIiIKICAgICAgICBsb2dpdHMgPSBzZWxmLm1vZGVsKHNlcXMpLmxvZ2l0cy5mbG9hdCgpCiAgICAgICAgcHJvYnMgPSBGLnNvZnRtYXgobG9naXRzLCBkaW09LTEpCiAgICAgICAgbWF4cCwgXyA9IHByb2JzLm1heChkaW09LTEpICAjIChCLCBMKQogICAgICAgIG91dCA9IFtdCiAgICAgICAgZm9yIGIgaW4gcmFuZ2Uoc2Vxcy5zaGFwZVswXSk6CiAgICAgICAgICAgIHNwYW4gPSBbXQogICAgICAgICAgICBmb3IgdCBpbiByYW5nZShwcm9tcHRfbGVuLCBzZXFzLnNoYXBlWzFdKToKICAgICAgICAgICAgICAgIHRvayA9IGludChzZXFzW2IsIHRdKQogICAgICAgICAgICAgICAgaWYgdG9rIGluIHNlbGYuZW9zX2lkcyBvciB0b2sgPT0gc2VsZi5tYXNrX3Rva2VuX2lkOgogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBzcGFuLmFwcGVuZChmbG9hdChtYXhwW2IsIHRdKSkKICAgICAgICAgICAgaWYgc3BhbjoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoKHN1bShzcGFuKSAvIGxlbihzcGFuKSwgbWluKHNwYW4pKSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoKDAuMCwgMC4wKSkKICAgICAgICByZXR1cm4gb3V0CgogICAgIyAtLS0tIG1haW4gZW50cnkgcG9pbnQgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiByZXNwb25kKAogICAgICAgIHNlbGYsCiAgICAgICAgcGVyc29uYTogUGVyc29uYSwKICAgICAgICB3b3JsZF9zdGF0ZTogc3RyLAogICAgICAgIHBsYXllcl9saW5lOiBzdHIsCiAgICAgICAgaGlzdG9yeTogbGlzdCB8IE5vbmUgPSBOb25lLAogICAgICAgIG5fY2FuZGlkYXRlczogaW50ID0gOCwKICAgICAgICBzdGVwczogaW50ID0gMzIsCiAgICAgICAgbWF4X25ld190b2tlbnM6IGludCA9IDY0LAogICAgICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9IDAuOCwKICAgICAgICB0b3BfcDogZmxvYXQgPSAwLjk1LAogICAgICAgIGNvbmZpZGVuY2VfdGhyZXNob2xkOiBmbG9hdCA9IDAuNDUsCiAgICApIC0+IE5QQ1Jlc3BvbnNlOgogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHByb21wdF9pZHMgPSBzZWxmLl9wcm9tcHRfaWRzKHBlcnNvbmEsIHdvcmxkX3N0YXRlLCBwbGF5ZXJfbGluZSwgaGlzdG9yeSkKICAgICAgICBwcm9tcHRfbGVuID0gcHJvbXB0X2lkcy5zaGFwZVsxXQoKICAgICAgICBjYW52YXMgPSB0b3JjaC5mdWxsKAogICAgICAgICAgICAoMSwgbWF4X25ld190b2tlbnMpLCBzZWxmLm1hc2tfdG9rZW5faWQsIGR0eXBlPXRvcmNoLmxvbmcsIGRldmljZT1zZWxmLmRldmljZQogICAgICAgICkKICAgICAgICBpbnB1dF9pZHMgPSB0b3JjaC5jYXQoW3Byb21wdF9pZHMsIGNhbnZhc10sIGRpbT0xKQogICAgICAgICMgcGFyYWxsZWwgY2FuZGlkYXRlczogTiBjYW52YXNlcywgT05FIGJhdGNoZWQgZGVub2lzaW5nIHJ1bgogICAgICAgIGlucHV0X2lkcyA9IGlucHV0X2lkcy5yZXBlYXQobl9jYW5kaWRhdGVzLCAxKQogICAgICAgIGF0dG4gPSB0b3JjaC5vbmVzX2xpa2UoaW5wdXRfaWRzKQoKICAgICAgICBnZW4gPSBzZWxmLm1vZGVsLmRpZmZ1c2lvbl9nZW5lcmF0ZSBpZiBoYXNhdHRyKHNlbGYubW9kZWwsICJkaWZmdXNpb25fZ2VuZXJhdGUiKSBlbHNlIHNlbGYubW9kZWwuZ2VuZXJhdGUKICAgICAgICBvdXQgPSBnZW4oCiAgICAgICAgICAgIGlucHV0X2lkcywKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2s9YXR0biwKICAgICAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgICAgIHN0ZXBzPXN0ZXBzLAogICAgICAgICAgICB0ZW1wZXJhdHVyZT10ZW1wZXJhdHVyZSwKICAgICAgICAgICAgdG9wX3A9dG9wX3AsCiAgICAgICAgICAgIHJldHVybl9kaWN0X2luX2dlbmVyYXRlPVRydWUsCiAgICAgICAgKQogICAgICAgIHNlcXMgPSBvdXQuc2VxdWVuY2VzIGlmIGhhc2F0dHIob3V0LCAic2VxdWVuY2VzIikgZWxzZSBvdXQKCiAgICAgICAgc2NvcmVzID0gc2VsZi5fc2NvcmVfY2FuZGlkYXRlcyhzZXFzLCBwcm9tcHRfbGVuKQogICAgICAgIHJhbmtlZCA9IFtdCiAgICAgICAgZm9yIGIgaW4gcmFuZ2Uoc2Vxcy5zaGFwZVswXSk6CiAgICAgICAgICAgIHRleHQgPSBzZWxmLl9kZWNvZGVfbGluZShzZXFzW2JdLCBwcm9tcHRfbGVuKQogICAgICAgICAgICBpZiB0ZXh0OgogICAgICAgICAgICAgICAgcmFua2VkLmFwcGVuZCgodGV4dCwgc2NvcmVzW2JdWzBdLCBzY29yZXNbYl1bMV0pKQogICAgICAgICMgZGVkdXBlIGlkZW50aWNhbCBsaW5lcywgcmFuayBieSBtZWFuIGNvbmZpZGVuY2UgdGhlbiB3ZWFrZXN0IGxpbmsKICAgICAgICBzZWVuLCB1bmlxID0gc2V0KCksIFtdCiAgICAgICAgZm9yIHRleHQsIG1lYW5fYywgbWluX2MgaW4gc29ydGVkKHJhbmtlZCwga2V5PWxhbWJkYSByOiAoclsxXSwgclsyXSksIHJldmVyc2U9VHJ1ZSk6CiAgICAgICAgICAgIGlmIHRleHQgbm90IGluIHNlZW46CiAgICAgICAgICAgICAgICBzZWVuLmFkZCh0ZXh0KQogICAgICAgICAgICAgICAgdW5pcS5hcHBlbmQoKHRleHQsIG1lYW5fYykpCiAgICAgICAgbGF0ZW5jeV9tcyA9ICh0aW1lLnBlcmZfY291bnRlcigpIC0gdDApICogMTAwMAoKICAgICAgICBpZiBub3QgdW5pcSBvciB1bmlxWzBdWzFdIDwgY29uZmlkZW5jZV90aHJlc2hvbGQ6CiAgICAgICAgICAgIGltcG9ydCByYW5kb20KICAgICAgICAgICAgZmFsbGJhY2sgPSByYW5kb20uY2hvaWNlKHBlcnNvbmEuZmFsbGJhY2tfbGluZXMpIGlmIHBlcnNvbmEuZmFsbGJhY2tfbGluZXMgZWxzZSAiLi4uIgogICAgICAgICAgICByZXR1cm4gTlBDUmVzcG9uc2UoCiAgICAgICAgICAgICAgICB0ZXh0PWZhbGxiYWNrLCBjb25maWRlbmNlPTEuMCwgdXNlZF9mYWxsYmFjaz1UcnVlLAogICAgICAgICAgICAgICAgY2FuZGlkYXRlcz11bmlxLCBsYXRlbmN5X21zPWxhdGVuY3lfbXMsCiAgICAgICAgICAgICkKICAgICAgICByZXR1cm4gTlBDUmVzcG9uc2UoCiAgICAgICAgICAgIHRleHQ9dW5pcVswXVswXSwgY29uZmlkZW5jZT11bmlxWzBdWzFdLCB1c2VkX2ZhbGxiYWNrPUZhbHNlLAogICAgICAgICAgICBjYW5kaWRhdGVzPXVuaXEsIGxhdGVuY3lfbXM9bGF0ZW5jeV9tcywKICAgICAgICApCgogICAgZGVmIF9kZWNvZGVfbGluZShzZWxmLCBzZXE6IHRvcmNoLlRlbnNvciwgcHJvbXB0X2xlbjogaW50KSAtPiBzdHI6CiAgICAgICAgaWRzID0gW10KICAgICAgICBmb3IgdCBpbiByYW5nZShwcm9tcHRfbGVuLCBzZXEuc2hhcGVbMF0pOgogICAgICAgICAgICB0b2sgPSBpbnQoc2VxW3RdKQogICAgICAgICAgICBpZiB0b2sgaW4gc2VsZi5lb3NfaWRzIG9yIHRvayA9PSBzZWxmLm1hc2tfdG9rZW5faWQ6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBpZHMuYXBwZW5kKHRvaykKICAgICAgICByZXR1cm4gc2VsZi50b2tlbml6ZXIuZGVjb2RlKGlkcywgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKS5zdHJpcCgpCg==",
    "npc_engine/demo.py": "IiIiSW50ZXJhY3RpdmUgdGV4dC1hZHZlbnR1cmUgZGVtbyBmb3IgdGhlIE5QQyBFbmdpbmUuCgpUd28gbW9kZXM6CiAgcHl0aG9uIC1tIG5wY19lbmdpbmUuZGVtbyAgICAgICAgICAgICMgdGFsayB0byB0aGUgTlBDcywgc2VlIGNhbmRpZGF0ZXMgKyBjb25maWRlbmNlcwogIHB5dGhvbiAtbSBucGNfZW5naW5lLmRlbW8gLS1iZW5jaCAgICAjIGxhdGVuY3kgYmVuY2htYXJrOiAyMCB0dXJucywgcmVwb3J0cyBwNTAvcDk1CiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBzdGF0aXN0aWNzCmltcG9ydCBzeXMKCmZyb20gLmVuZ2luZSBpbXBvcnQgTlBDRW5naW5lCmZyb20gLnBlcnNvbmFzIGltcG9ydCBTVEFSVEVSX1BFUlNPTkFTCgpXT1JMRF9TVEFURVMgPSBbCiAgICAiRXZlbmluZy4gVGhlIG1pbGxlcidzIGNhcnQgdGhyZXcgYSB3aGVlbCBvbiB0aGUgbm9ydGggcm9hZDsgYmFuZGl0cyB3ZXJlIHNlZW4gdGhlcmUgbGFzdCB3ZWVrLiIsCiAgICAiTW9ybmluZyBtYXJrZXQgZGF5LiBUaGUgc3F1YXJlIGlzIGJ1c3k7IGEgdHJhdmVsZXIgd2FzIHJvYmJlZCB5ZXN0ZXJkYXkuIiwKICAgICJMYXRlIG5pZ2h0LiBSYWluLiBUaGUgdGF2ZXJuIGlzIG5lYXJseSBlbXB0eTsgYSBzdHJhbmdlciB3YXRjaGVzIGZyb20gdGhlIGNvcm5lci4iLApdCgoKZGVmIGludGVyYWN0aXZlKGVuZ2luZTogTlBDRW5naW5lKToKICAgIHByaW50KCI9PT0gTlBDIEVuZ2luZSDigJQgdGV4dCBhZHZlbnR1cmUgZGVtbyA9PT1cbiIpCiAgICBmb3IgaSwgcCBpbiBlbnVtZXJhdGUoU1RBUlRFUl9QRVJTT05BUyk6CiAgICAgICAgcHJpbnQoZiIgIFt7aX1dIHtwLm5hbWV9IOKAlCB7cC5yb2xlfSIpCiAgICB0cnk6CiAgICAgICAgY2hvaWNlID0gaW50KGlucHV0KCJcblBpY2sgYW4gTlBDICgwLTUpOiAiKS5zdHJpcCgpIG9yICIwIikKICAgICAgICBwZXJzb25hID0gU1RBUlRFUl9QRVJTT05BU1tjaG9pY2UgJSBsZW4oU1RBUlRFUl9QRVJTT05BUyldCiAgICBleGNlcHQgKFZhbHVlRXJyb3IsIEluZGV4RXJyb3IpOgogICAgICAgIHBlcnNvbmEgPSBTVEFSVEVSX1BFUlNPTkFTWzBdCiAgICB3b3JsZCA9IFdPUkxEX1NUQVRFU1swXQogICAgcHJpbnQoZiJcbllvdSBhcHByb2FjaCB7cGVyc29uYS5uYW1lfS4gKHt3b3JsZH0pIikKICAgIHByaW50KCJUeXBlIHRvIHRoZW0uICdxdWl0JyB0byBsZWF2ZSwgJ25wYycgdG8gc3dpdGNoIGNoYXJhY3Rlci5cbiIpCiAgICB3aGlsZSBUcnVlOgogICAgICAgIHRyeToKICAgICAgICAgICAgbGluZSA9IGlucHV0KCJ5b3U+ICIpLnN0cmlwKCkKICAgICAgICBleGNlcHQgKEVPRkVycm9yLCBLZXlib2FyZEludGVycnVwdCk6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgaWYgbm90IGxpbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgbGluZS5sb3dlcigpID09ICJxdWl0IjoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBsaW5lLmxvd2VyKCkgPT0gIm5wYyI6CiAgICAgICAgICAgIHJldHVybiBpbnRlcmFjdGl2ZShlbmdpbmUpCiAgICAgICAgciA9IGVuZ2luZS5yZXNwb25kKHBlcnNvbmEsIHdvcmxkLCBsaW5lKQogICAgICAgIGZsYWcgPSAiIFtGQUxMQkFDS10iIGlmIHIudXNlZF9mYWxsYmFjayBlbHNlICIiCiAgICAgICAgcHJpbnQoZiJcbntwZXJzb25hLm5hbWV9PiB7ci50ZXh0fXtmbGFnfSIpCiAgICAgICAgcHJpbnQoZiIgIChjb25maWRlbmNlIHtyLmNvbmZpZGVuY2U6LjJmfSB8IHtyLmxhdGVuY3lfbXM6LjBmfSBtcyB8ICIKICAgICAgICAgICAgICBmIntsZW4oci5jYW5kaWRhdGVzKX0gdW5pcXVlIGNhbmRpZGF0ZXMpIikKICAgICAgICBpZiBsZW4oci5jYW5kaWRhdGVzKSA+IDE6CiAgICAgICAgICAgIGFsdHMgPSAiLCAiLmpvaW4oZicie3R9IiAoe2M6LjJmfSknIGZvciB0LCBjIGluIHIuY2FuZGlkYXRlc1sxOjRdKQogICAgICAgICAgICBwcmludChmIiAgYWx0ZXJuYXRpdmVzOiB7YWx0c30iKQogICAgICAgIHByaW50KCkKCgpkZWYgYmVuY2htYXJrKGVuZ2luZTogTlBDRW5naW5lKToKICAgIHBlcnNvbmEgPSBTVEFSVEVSX1BFUlNPTkFTWzBdCiAgICB3b3JsZCA9IFdPUkxEX1NUQVRFU1swXQogICAgcHJvYmVzID0gWwogICAgICAgICJBbnkgd29yayBmb3IgbWU/IiwKICAgICAgICAiV2hhdCBoYXBwZW5lZCBvbiB0aGUgbm9ydGggcm9hZD8iLAogICAgICAgICJJIG5lZWQgYSBzd29yZCwgZmFzdC4iLAogICAgICAgICJUZWxsIG1lIGFib3V0IHRoZSBtaWxsZXIuIiwKICAgICAgICAiQXJlIHRoZSBiYW5kaXRzIHN0aWxsIG91dCB0aGVyZT8iLAogICAgXSAqIDQKICAgIGxhdCwgY29uZiwgZmIgPSBbXSwgW10sIDAKICAgIHByaW50KGYicnVubmluZyB7bGVuKHByb2Jlcyl9IHR1cm5zLCA4IGNhbmRpZGF0ZXMgZWFjaC4uLiIpCiAgICBmb3IgcCBpbiBwcm9iZXM6CiAgICAgICAgciA9IGVuZ2luZS5yZXNwb25kKHBlcnNvbmEsIHdvcmxkLCBwKQogICAgICAgIGxhdC5hcHBlbmQoci5sYXRlbmN5X21zKQogICAgICAgIGNvbmYuYXBwZW5kKHIuY29uZmlkZW5jZSkKICAgICAgICBmYiArPSByLnVzZWRfZmFsbGJhY2sKICAgIGxhdC5zb3J0KCkKICAgIHA1MCA9IHN0YXRpc3RpY3MubWVkaWFuKGxhdCkKICAgIHA5NSA9IGxhdFtpbnQoMC45NSAqIGxlbihsYXQpKSAtIDFdCiAgICBwcmludChmIlxucDUwIGxhdGVuY3k6IHtwNTA6LjBmfSBtcyAgIHA5NSBsYXRlbmN5OiB7cDk1Oi4wZn0gbXMiKQogICAgcHJpbnQoZiJtZWFuIGNvbmZpZGVuY2U6IHtzdGF0aXN0aWNzLm1lYW4oY29uZik6LjJmfSAgIGZhbGxiYWNrczoge2ZifS97bGVuKHByb2Jlcyl9IikKICAgIHByaW50KGYidGhyb3VnaHB1dDogezEwMDAgLyBwNTAgKiA4Oi4wZn0gY2FuZGlkYXRlLWxpbmVzL3NlYyAoOCBwZXIgdHVybikiKQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iZW5jaCIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbW9kZWwiLCBkZWZhdWx0PSJEcmVhbS1vcmcvRHJlYW0tdjAtSW5zdHJ1Y3QtN0IiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLThiaXQiLCBkZXN0PSJlaWdodGJpdCIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgICAgICAgICAgICAgaGVscD0ibG9hZCBpbiA4LWJpdCAoZm9yIDE2R0IgR1BVcykiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQogICAgcHJpbnQoImxvYWRpbmcgbW9kZWwgKGZpcnN0IHJ1biBkb3dubG9hZHMgfjE0R0IpLi4uIiwgZmx1c2g9VHJ1ZSkKICAgIGVuZ2luZSA9IE5QQ0VuZ2luZShtb2RlbF9pZD1hcmdzLm1vZGVsLCBsb2FkX2luXzhiaXQ9YXJncy5laWdodGJpdCkKICAgIGlmIGFyZ3MuYmVuY2g6CiAgICAgICAgYmVuY2htYXJrKGVuZ2luZSkKICAgIGVsc2U6CiAgICAgICAgaW50ZXJhY3RpdmUoZW5naW5lKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkpCg==",
}
for path, b64 in files.items():
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
print("npc_engine package restored:", sorted(str(p) for p in pathlib.Path("npc_engine").glob("*.py")))


## Load the model (8-bit fits a T4 16GB)

In [ ]:
from npc_engine.engine import NPCEngine
from npc_engine.personas import STARTER_PERSONAS
import npc_engine
print('npc_engine version:', npc_engine.__version__)  # must say 0.1.2 or later

engine = NPCEngine(load_in_8bit=True)
print('ready on', engine.device)


## Scripted conversation — 3 NPCs, watch candidates + confidences

In [ ]:
scenes = [
    (0, "Evening. The miller's cart threw a wheel on the north road; bandits were seen there last week.",
     ["Any work for me?", "What happened on the north road?"]),
    (1, "Morning market day. The square is busy; a traveler was robbed yesterday.",
     ["What's good today?", "Heard any gossip?"]),
    (2, "Late night. Rain. The tavern is nearly empty.",
     ["I need to pass the north gate.", "Are the bandits still out there?"]),
]
for idx, world, lines in scenes:
    p = STARTER_PERSONAS[idx]
    print(f"\n===== {p.name} ({p.role}) =====")
    for line in lines:
        r = engine.respond(p, world, line)
        flag = ' [FALLBACK]' if r.used_fallback else ''
        print(f"\nyou> {line}")
        print(f"{p.name}> {r.text}{flag}  (conf {r.confidence:.2f}, {r.latency_ms:.0f} ms)")
        for t, c in r.candidates[1:3]:
            print(f"    alt ({c:.2f}): {t}")


## Benchmark — 20 turns, latency distribution

In [ ]:
import statistics
p = STARTER_PERSONAS[0]
world = "Evening. The miller's cart threw a wheel on the north road."
probes = ["Any work for me?", "What happened on the north road?", "I need a sword, fast.",
          "Tell me about the miller.", "Are the bandits still out there?"] * 4
lat, conf, fb = [], [], 0
for q in probes:
    r = engine.respond(p, world, q)
    lat.append(r.latency_ms); conf.append(r.confidence); fb += r.used_fallback
lat.sort()
print(f"p50 {statistics.median(lat):.0f} ms | p95 {lat[int(0.95*len(lat))-1]:.0f} ms | "
      f"mean conf {statistics.mean(conf):.2f} | fallbacks {fb}/{len(probes)}")


## Reading the results

- **p50 under ~1s** on a T4 with 8-bit = the concept works; a 5090/4090 at fp16 will be several times faster.
- **Confidence vs quality**: if lines that read wrong score low and good lines score high, the ranking signal is honest — that's the anti-gameslop guard.
- **Next steps after this**: fine-tune Dream 7B (LoRA) on persona-conditioned dialogue (agentlans triples + LIGHT + synthetic), then the Unity plugin.
